In [1]:
import pandas as pd
from PIL import Image
# import model and dataset for both PNG and embed dicom images
from modules.spot_compression import *
import torchvision.transforms as T
import torch
import os
from torch.utils.data import DataLoader

In [ ]:
ROOT = ''

img_dir = f'{ROOT}/sample_data/images'

val_transform = T.Compose([
        T.Grayscale(num_output_channels=1),
        T.Resize(224), 
        T.CenterCrop((224, 224)), 
        T.ToTensor(),
    ])

sample_dataset = ImageDatasetEMBED('sample_dataset.csv', val_transform)

[sample_dataset.csv] Loaded 6, pos(1)=1, neg(0)=5


In [3]:
CHECKPOINT_PATH = "weights/spot_comp_model.ckpt" 
GPU_INDEX       = 7

BEST_PARAMS = {
    'model': 'resnet50', 
    'batch_size': 32,
    'lr': 1e-4, 
    'weight_decay': 1e-5,
    'dropout_p': 0.5,
    'rotation_degrees': 90,
    'crop_size': 224
}

model = ImageClassifier.load_from_checkpoint(
    CHECKPOINT_PATH,
    model_name=BEST_PARAMS['model'],
    lr=BEST_PARAMS['lr'],
    pretrained=False,
    weight_decay=BEST_PARAMS['weight_decay'],
    pos_weight=torch.tensor([1.0]),
    dropout_p=BEST_PARAMS['dropout_p']
    )
    
model.eval()
model.freeze()

ImageClassifier(
  (model): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
    

In [4]:

all_probs = []
all_labels = []
all_paths = []

dataloader = DataLoader(sample_dataset, 
                        batch_size=BEST_PARAMS['batch_size'], 
                        shuffle=False, 
                        num_workers=8, 
                        pin_memory=True)

device = torch.device(f"cuda:{GPU_INDEX}" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    for batch in dataloader:
        p, x, y = batch
        logits = model(x.to(device)).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.numpy())
        all_paths.extend(p)

df = pd.DataFrame({
    'file_name': all_paths,
    'true_label': all_labels,
    'probability': all_probs
 })

In [5]:
df

,file_name,true_label,probability
0,sample_data/sample_1.png,0.0,0.442923
1,sample_data/sample_2.png,0.0,0.446250
2,sample_data/sample_3.png,0.0,0.452080
3,sample_data/sample_4.png,0.0,0.550717
4,sample_data/sample_5.png,1.0,0.971482
5,sample_data/sample_6.png,0.0,0.437619
